### [W-okada's Deiteris' tg-develops' Fork Realtime Voice Changer](https://github.com/tg-develop/voice-changer) | **Lightning.AI**

---

## **⬇ VERY IMPORTANT ⬇**

You can use the following settings for optimal results:

Best performance: `f0: fcpe | Chunk: 56.0ms or higher | Extra: 2.7s`<br>
Best quality: `f0: rmvpe | Chunk: 64.0ms or higher | Extra: 5s`<br>
Don't forget to select your Lightning.AI GPU in the GPU field, <b>NEVER</b> use CPU!

You can tune `Chunk` for lower/higher delay and `Extra` for better quality.

**NOTE: ⚠️ "Server audio not available" is normal on cloud platforms as they have no physical sound card. Always use Client mode for audio input/output.**

<br>

---

### Always use Lightning.AI GPU (**VERY VERY VERY IMPORTANT!**)
You need to use a Lightning.AI GPU so the Voice Changer can work faster and better.
Use the lateral right menu, and click on **Studio Environment (the Processor Icon)** » **Switch To GPU** » select a GPU, **check [pricing](https://lightning.ai/pricing)** for more info about how many hours you can use each GPU and which is free.

---

# **Credits and Support**
Original W-okada Realtime Voice Changer by [w-okada](https://github.com/w-okada)<br>
W-okada Deiteris Fork Realtime Voice Changer used as a base for tg-develop by [deiteris](https://github.com/deiteris)<br>
Wokada tg-develop Fork Realtime Voice Changer by [tg-develop](https://github.com/tg-develop)<br>
Original instructions by [Hina](https://github.com/HinaBl)<br>
Tunnels (Port Viewer, Cloudflare, LocalTunnel, Horizon), Decryption and Lightning.AI Port by [Nick088](https://linktr.ee/Nick088)<br>

Need help? Ask in the [AI Hub Discord](https://discord.gg/aihub)

---

# Clone repository and install dependencies
This first step will download the latest version of tg-develops' Voice Changer and install the dependencies. **It can take some time to complete, but after you do it the first time it will persist.**

## (Optional) ClearConsole:
You can set `ClearConsole` to `True` or `False` whether you want to clear the output when done or not.

`Default: True`

In [ ]:
ClearConsole = True  # @param {type:"boolean"}



import os
import time
from IPython.display import clear_output

%cd /teamspace/studios/this_studio

!pip install colorama python-dotenv --quiet

from colorama import Fore, Style
import requests

LATEST_RELEASE = "https://api.github.com/repos/tg-develop/voice-changer/releases/latest"
LINUX_PREBUILT = "voice-changer-linux-amd64-cuda.tar.gz"


def download_and_unpack(assets: list):
    for asset in assets:
        if not asset['name'].startswith(LINUX_PREBUILT):
            continue
        download_url = asset['browser_download_url']
        !wget -q --show-progress {download_url}

    print(f"{Fore.GREEN}> Unpacking...{Style.RESET_ALL}")
    !cat voice-changer-linux-amd64-cuda.tar.gz.* | tar xzf -
    print(f"{Fore.GREEN}> Finished unpacking!{Style.RESET_ALL}")
    !rm -rf voice-changer-linux-amd64-cuda.tar.gz.*

    !mv /teamspace/studios/this_studio/MMVCServerSIO/MMVCServerSIO /teamspace/studios/this_studio/MMVCServerSIO/VoiceChanger


def get_local_version(base: str) -> str:
    local_version = !cat {base}/_internal/version.txt
    return local_version[0].strip()


print(f"{Fore.CYAN}> Downloading prebuilt executable...{Style.RESET_ALL}")

res = requests.get(LATEST_RELEASE)
release_info = res.json()

remote_version = release_info['tag_name']

# Check and upgrade if already installed
if os.path.exists('/teamspace/studios/this_studio/MMVCServerSIO'):
    local_version = get_local_version("/teamspace/studios/this_studio/MMVCServerSIO")
    if remote_version != local_version:
        print(f"{Fore.CYAN}> Found new version. Current version {local_version}, remote version {remote_version}. Upgrading...{Style.RESET_ALL}")
        !rm -rf /teamspace/studios/this_studio/MMVCServerSIO/_internal /teamspace/studios/this_studio/MMVCServerSIO/VoiceChanger
        download_and_unpack(release_info['assets'])
    else:
        print(f"{Fore.CYAN}> Current version {local_version} is the latest. Skipping download.{Style.RESET_ALL}")
else:
    print(f"{Fore.CYAN}> Downloading and installing version {remote_version}.{Style.RESET_ALL}")
    download_and_unpack(release_info['assets'])

%cd /teamspace/studios/this_studio

print(f"{Fore.GREEN}> Successfully downloaded and unpacked the binary!{Style.RESET_ALL}")

print(f"{Fore.CYAN}> Installing libportaudio2...{Style.RESET_ALL}")
!sudo apt-get update -y
!sudo apt-get -y install libportaudio2
# Keep psmisc on persistent sessions to kill previous instances
!sudo apt-get install psmisc

# Fix: replace the bundled libstdc++.so.6 inside _internal with the system one.
# The bundled version is too old and causes a GLIBCXX_3.4.32 error when loading
# libportaudio via libjack on Lightning.AI's environment.
print(f"{Fore.CYAN}> Applying libstdc++ fix...{Style.RESET_ALL}")
!SYSTEM_LIBSTDCPP=$(ldconfig -p | grep 'libstdc++.so.6' | grep 'x86-64' | awk '{print $NF}' | head -1) && \
 cp "$SYSTEM_LIBSTDCPP" /teamspace/studios/this_studio/MMVCServerSIO/_internal/libstdc++.so.6
print(f"{Fore.GREEN}> libstdc++ fix applied!{Style.RESET_ALL}")

if ClearConsole:
    clear_output()

print("Installed!")

# Set server configuration
This cell will apply the necessary configuration to the server.

In [ ]:
%cd /teamspace/studios/this_studio/MMVCServerSIO

from dotenv import set_key

set_key('.env', "SAMPLE_MODE", "")

Ready = True

print("Server successfully configured!")

# Start Server **using Tunnels**
This cell will start the server via Tunnels. The first time you run it, it will download the models, so it can take a few minutes (~1-2 minutes).

## Tunnels

Select the type of tunnel you want to use for the public link. If one is down, you can switch to another:

 **A) Port Viewer:** Select it in Tunnel, Click the **+** at the bottom of the right tab, click on **Web Apps** and install **Port Viewer**, run the cell, then wait for the Local URL to appear and click on Port Viewer on the right tab, add a new port and write **18888** as the Port Number and optionally give it a name. After it's added, double click on the port name to see the UI, and to see it bigger you can click **Open**.

 **B) Ngrok:** Select it in Tunnel, get the Ngrok Tunnel Authtoken here: https://dashboard.ngrok.com/tunnels/authtokens/new, paste it in the `Token` field, run the cell, and wait for the Ngrok Tunnel Public URL to appear. It has a 1GB Bandwidth Free Monthly Limit.

 **C) Cloudflare:** Select it in Tunnel, run the cell, and wait for the Cloudflare Tunnel Public URL to appear.

 **D) LocalTunnel:** Select it in Tunnel, run the cell, wait for the Local URL to appear, copy the **LocalTunnel Password** displayed under the public link and paste it in the Tunnel Password field at the LocalTunnel URL.

 **E) Horizon:** Select it in Tunnel, get your Horizon ID here: https://hrzn.run/dashboard/, on the 2nd step it shows `hrzn login YOURID` — copy that ID and put it in the `Token` field. Run the cell, you will get `HORIZON: Authorize at https://hrzn.run/dashboard/settings/cli-token-requests/YOUR_ID` — click it and approve. Then wait for the Horizon Tunnel Public URL to appear.

## (Optional) Ngrok Region
If you're using the Ngrok Tunnel, you can change to a region near to you.

`Default Region: eu - Europe (Frankfurt)`

## (Optional) ClearConsole
Set `ClearConsole` to `True` or `False` whether you want to clear the output when the server is ready or not.

`Default: True`

In [ ]:
#======================Tunnels===========================

Tunnel = "Port Viewer" #@param ["Port Viewer", "Ngrok", "Cloudflare", "LocalTunnel", "Horizon"]

Token = 'Ngrok | Horizon TOKEN' # @param {type:"string"}
Region = "eu - Europe (Frankfurt)" # @param ["ap - Asia/Pacific (Singapore)", "au - Australia (Sydney)", "eu - Europe (Frankfurt)", "in - India (Mumbai)", "jp - Japan (Tokyo)", "sa - South America (Sao Paulo)", "us - United States (Ohio)"]

ClearConsole = True  # @param {type:"boolean"}



if not globals().get('Ready', False):
    raise RuntimeError("Go back and run the first and second cells before starting the server.")

# Fix issues to keep portaudio on persistent sessions
!sudo apt-get update -y
!sudo apt-get -y install libportaudio2
# Kill any previous instances on port 18888
!sudo apt-get install psmisc
!fuser -k 18888/tcp

PORT = 18888

import json
import threading, time, socket
from IPython.display import clear_output
from dotenv import set_key

%cd /teamspace/studios/this_studio/

if Tunnel == "Port Viewer":
    print("Be sure to run the manual steps in the guide above for Port Viewer!")
    import urllib
    import subprocess
    endpoint_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()

    # Construct the origin URL from the endpoint IP
    ip_origin = f"http://{endpoint_ip}:{PORT}"

    # Auto-detect the Lightning.AI cloudspaces URL for this port
    try:
        result = subprocess.run(
            ["bash", "-c", "echo $LIGHTNING_CLOUDSPACE_HOST"],
            capture_output=True, text=True
        )
        cloudspace_host = result.stdout.strip()
        if cloudspace_host:
            cloudspaces_origin = f"https://{PORT}-{cloudspace_host}"
        else:
            cloudspaces_origin = None
    except Exception:
        cloudspaces_origin = None

    public_url = "Please use the Port Viewer URL from the Lightning AI UI explained in the guide above."

elif Tunnel == "Ngrok":
    !pip install pyngrok --quiet
    from pyngrok import conf, ngrok
    MyConfig = conf.PyngrokConfig()
    MyConfig.auth_token = Token
    MyConfig.region = Region[0:2]
    conf.get_default().authtoken = Token
    conf.get_default().region = Region
    conf.set_default(MyConfig)

    from pyngrok import ngrok

    ngrokConnection = ngrok.connect(PORT)
    public_url = ngrokConnection.public_url

elif Tunnel == "Cloudflare":
    !curl -LO https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !sudo dpkg -i cloudflared-linux-amd64.deb
    import time
    import subprocess

    command = ["cloudflared", "tunnel", "--url", f"localhost:{PORT}"]

    with open("nohup.out", "w") as f:
        proc = subprocess.Popen(command, stdout=f, stderr=subprocess.STDOUT)

    # Wait until the Cloudflare URL appears in nohup.out, up to 20 seconds
    for _ in range(40):
        time.sleep(0.5)
        with open('nohup.out', 'r') as file:
            content = file.read()
        if 'trycloudflare.com' in content:
            break
    else:
        raise RuntimeError("Cloudflare tunnel failed to start within 20 seconds. Check your connection, try again or try other tunnel options.")

    cloudflare_url = !grep -oE "https://[a-zA-Z0-9.-]+\.trycloudflare\.com" nohup.out
    if cloudflare_url:
        public_url = cloudflare_url[0]

elif Tunnel == "LocalTunnel":
    !sudo apt-get update -y
    !sudo apt-get install -y curl

    !curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash -

    !sudo apt-get install -y nodejs
    print("Node.js & npm installation complete!")

    !sudo npm install -g localtunnel
    import time
    import urllib

    with open('/teamspace/studios/this_studio/url.txt', 'w') as file:
        file.write('')

    get_ipython().system_raw(f'lt --port {PORT} >> /teamspace/studios/this_studio/url.txt 2>&1 &')

    # Wait until the LocalTunnel URL appears, up to 20 seconds
    for _ in range(40):
        time.sleep(0.5)
        with open('/teamspace/studios/this_studio/url.txt', 'r') as file:
            content = file.read()
        if 'loca.lt' in content:
            break
    else:
        raise RuntimeError("LocalTunnel failed to start within 20 seconds. Check your connection, try again or try other tunnel options.")

    endpoint_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n")

    with open('/teamspace/studios/this_studio/url.txt', 'r') as file:
        public_url = file.read()
        public_url = public_url.replace("your url is: ", "").strip()

elif Tunnel == "Horizon":
    import os
    !sudo apt-get update -y
    !sudo apt-get install -y curl

    !curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash -

    !sudo apt-get install -y nodejs
    print("Node.js & npm installation complete!")

    !sudo npm install -g @hrzn/cli
    !hrzn login $Token
    os.system(f"hrzn tunnel http://localhost:{PORT} >> /teamspace/studios/this_studio/url.txt 2>&1 &")

    # Wait until the Horizon URL appears, up to 20 seconds
    import time
    for _ in range(40):
        time.sleep(0.5)
        with open('/teamspace/studios/this_studio/url.txt', 'r') as file:
            content = file.read()
        if 'hrzn.run' in content:
            break
    else:
        raise RuntimeError("Horizon tunnel failed to start within 20 seconds. Check your token and try again.")

    public_url = !grep -oE "https://[a-zA-Z0-9.-]+\.hrzn\.run" /teamspace/studios/this_studio/url.txt
    public_url = public_url[0]

%cd /teamspace/studios/this_studio/MMVCServerSIO

# Set the allowed origins based on the tunnel type
if Tunnel == "Port Viewer":
    allowed_origins = [ip_origin]
    if cloudspaces_origin:
        allowed_origins.append(cloudspaces_origin)
else:
    allowed_origins = [public_url]

set_key('.env', "ALLOWED_ORIGINS", json.dumps(allowed_origins))


def wait_for_server():
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', PORT))
        if result == 0:
            sock.close()
            break

    if ClearConsole:
        clear_output()

    print("--------- SERVER READY! ---------")
    print("Your server is available at:")
    print(public_url)
    if Tunnel == "LocalTunnel":
        print("---------------------------------")
        print(f"Local Tunnel Password: {endpoint_ip}")
    print("---------------------------------")


threading.Thread(target=wait_for_server, daemon=True).start()

!chmod +x ./VoiceChanger

!./VoiceChanger

# Tunnels clean up
if Tunnel == "Ngrok":
    ngrok.disconnect(ngrokConnection.public_url)
elif Tunnel == "Cloudflare":
    !rm -rf nohup.out
    !rm -rf /teamspace/studios/this_studio/cloudflared-linux-amd64.deb
elif Tunnel == "LocalTunnel":
    !rm -rf /teamspace/studios/this_studio/url.txt
elif Tunnel == "Horizon":
    !rm -rf /teamspace/studios/this_studio/url.txt
    !fuser -k ${PORT}/tcp

print("--------- SERVER STOPPED! ---------")

# Delete everything

Lightning.AI sessions storage persists. If there are any updates, you need to delete everything first and then re-install it by running the first cell again.

In [ ]:
# Main program folder
!rm -rf /teamspace/studios/this_studio/MMVCServerSIO
# Cloudflare tunnel temp files
!rm -rf /teamspace/studios/this_studio/nohup.out
!rm -rf /teamspace/studios/this_studio/cloudflared-linux-amd64.deb
# LocalTunnel / Horizon temp files
!rm -rf /teamspace/studios/this_studio/url.txt

print("Successfully deleted everything!")